### Dependencies

In [ ]:
import openai
import pandas as pd  

from IPython.display import Image, display

from qdrant_client import QdrantClient, models
from qdrant_client.models import (
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    PointStruct,
    Document,
    Prefetch,
    Fusion,
    FusionQuery,
)
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END

### Single Node Graph

In [ ]:
# class StateMessage(BaseModel):
#   message: str

# class StateAnswer(BaseModel):
#   answer: str = ""


# class StateVibe(BaseModel):
#   vibe: str

# class State(StateMessage, StateAnswer, StateVibe):
#   pass

class State(BaseModel):
  message: str 
  answer: str = ""
  vibe: str


In [ ]:
from typing import TypedDict


# class AnswerContainer(TypedDict):
#   answer: str

def append_vibes_to_query(state: State) -> State:
  return state.model_copy(update={"answer": f"{state.message} {state.vibe}"})

In [ ]:
workflow = StateGraph(State)
workflow.add_node("append_vibes_to_query", append_vibes_to_query)
workflow.add_edge(START, "append_vibes_to_query")
workflow.add_edge("append_vibes_to_query", END)

graph = workflow.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = State(
  message="Give me some vibes",
  vibe="I'm feeling like a badass today!"
)

In [ ]:
result = graph.invoke(initial_state)

In [ ]:
result

In [ ]:
initial_state = {
  "message": "Give me some vibes",
  "answer": "abc",
  "vibe": "I'm feeling like a badass today!" 
}


In [ ]:
result

### Conditional Graph

In [ ]:
class State(BaseModel):
  message: str
  answer: str = ""

def append_vibes_to_query(state: State) -> State:
  return state.model_copy(update={"answer": state.message})

In [ ]:
def append_vibe_1(state: State) -> State:
  vibe = "I'm feeling like a badass today!"
  return state.model_copy(update={"answer": f"{state.message} {vibe}"})

def append_vibe_2(state: State) -> State:
  vibe = "I'm feeling like a boss today!"
  return state.model_copy(update={"answer": f"{state.message} {vibe}"})

def append_vibe_3(state: State) -> State:
  vibe = "I'm feeling like a legend today!"
  return state.model_copy(update={"answer": f"{state.message} {vibe}"})

In [ ]:
import random
from enum import StrEnum

class Nodes(StrEnum):
    APPEND_VIBES_TO_QUERY = "append_vibes_to_query"
    APPEND_VIBE_1 = "append_vibe_1"
    APPEND_VIBE_2 = "append_vibe_2"
    APPEND_VIBE_3 = "append_vibe_3"


VIBE_OPTIONS = (
    Nodes.APPEND_VIBE_1,
    Nodes.APPEND_VIBE_2,
    Nodes.APPEND_VIBE_3,
)

def router(state: State) -> Nodes:
  return random.choice(list(VIBE_OPTIONS))

workflow = StateGraph(State)
workflow.add_node(Nodes.APPEND_VIBES_TO_QUERY, append_vibes_to_query)
workflow.add_node(Nodes.APPEND_VIBE_1, append_vibe_1)
workflow.add_node(Nodes.APPEND_VIBE_2, append_vibe_2)
workflow.add_node(Nodes.APPEND_VIBE_3, append_vibe_3)

workflow.add_conditional_edges(
  Nodes.APPEND_VIBES_TO_QUERY,
  router,
  {member: member for member in VIBE_OPTIONS},
)

workflow.add_edge(START, Nodes.APPEND_VIBES_TO_QUERY)
workflow.add_edge(Nodes.APPEND_VIBE_1, END)
workflow.add_edge(Nodes.APPEND_VIBE_2, END)
workflow.add_edge(Nodes.APPEND_VIBE_3, END)

graph = workflow.compile()


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
initial_state = State(
  message="Give me some vibes",
)
result = graph.invoke(initial_state)
result


In [ ]:
def router(state: State) -> Nodes:
    return random.choice(list(VIBE_OPTIONS))


workflow = StateGraph(State)
workflow.add_node(Nodes.APPEND_VIBE_1, append_vibe_1)
workflow.add_node(Nodes.APPEND_VIBE_2, append_vibe_2)
workflow.add_node(Nodes.APPEND_VIBE_3, append_vibe_3)

workflow.add_conditional_edges(
    START,
    router,
    {member: member for member in VIBE_OPTIONS},
)

workflow.add_edge(Nodes.APPEND_VIBE_1, END)
workflow.add_edge(Nodes.APPEND_VIBE_2, END)
workflow.add_edge(Nodes.APPEND_VIBE_3, END)

graph = workflow.compile()


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = State(
    message="Give me some vibes",
)
result = graph.invoke(initial_state)
result
